# 02 - Data Preprocessing & Feature Preparation

## Objective
Notebook này thực hiện các bước tiền xử lý dữ liệu cho bài toán phân tích hiệu suất và nghỉ việc của nhân viên, bao gồm:
- Khảo sát và làm sạch dữ liệu
- Xử lý giá trị thiếu
- Loại bỏ các cột không mang thông tin
- Mã hóa biến phân loại
- Chuẩn hóa dữ liệu số
- Chuẩn bị dữ liệu cho các bước khai phá và mô hình hóa tiếp theo

In [1]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split

pd.set_option('display.max_columns', None)

In [2]:
DATA_PATH = "../data/raw/HR_Analytics.csv"

df = pd.read_csv(DATA_PATH)
df.head()

,EmpID,Age,AgeGroup,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeNumber,EnvironmentSatisfaction,Gender,HourlyRate,JobInvolvement,JobLevel,JobRole,JobSatisfaction,MaritalStatus,MonthlyIncome,SalarySlab,MonthlyRate,NumCompaniesWorked,Over18,OverTime,PercentSalaryHike,PerformanceRating,RelationshipSatisfaction,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,RM297,18,18-25,Yes,Travel_Rarely,230,Research & Development,3,3,Life Sciences,1,405,3,Male,54,3,1,Laboratory Technician,3,Single,1420,Upto 5k,25233,1,Y,No,13,3,3,80,0,0,2,3,0,0,0,0.0
1,RM302,18,18-25,No,Travel_Rarely,812,Sales,10,3,Medical,1,411,4,Female,69,2,1,Sales Representative,3,Single,1200,Upto 5k,9724,1,Y,No,12,3,1,80,0,0,2,3,0,0,0,0.0
2,RM458,18,18-25,Yes,Travel_Frequently,1306,Sales,5,3,Marketing,1,614,2,Male,69,3,1,Sales Representative,2,Single,1878,Upto 5k,8059,1,Y,Yes,14,3,4,80,0,0,3,3,0,0,0,0.0
3,RM728,18,18-25,No,Non-Travel,287,Research & Development,5,2,Life Sciences,1,1012,2,Male,73,3,1,Research Scientist,4,Single,1051,Upto 5k,13493,1,Y,No,15,3,4,80,0,0,2,3,0,0,0,0.0
4,RM829,18,18-25,Yes,Non-Travel,247,Research & Development,8,1,Medical,1,1156,3,Male,80,3,1,Laboratory Technician,3,Single,1904,Upto 5k,13556,1,Y,No,12,3,4,80,0,0,0,3,0,0,0,0.0


In [3]:
# Kiểm tra kích thước dữ liệu
print("Shape:", df.shape)

Shape: (1480, 38)


In [4]:
# Kiểm tra missing values
df.isnull().sum()

EmpID                        0
Age                          0
AgeGroup                     0
Attrition                    0
BusinessTravel               0
DailyRate                    0
Department                   0
DistanceFromHome             0
Education                    0
EducationField               0
EmployeeCount                0
EmployeeNumber               0
EnvironmentSatisfaction      0
Gender                       0
HourlyRate                   0
JobInvolvement               0
JobLevel                     0
JobRole                      0
JobSatisfaction              0
MaritalStatus                0
MonthlyIncome                0
SalarySlab                   0
MonthlyRate                  0
NumCompaniesWorked           0
Over18                       0
OverTime                     0
PercentSalaryHike            0
PerformanceRating            0
RelationshipSatisfaction     0
StandardHours                0
StockOptionLevel             0
TotalWorkingYears            0
Training

In [5]:
# Xử lý missing values
df['YearsWithCurrManager'].fillna(
    df['YearsWithCurrManager'].median(), inplace=True
)

df.isnull().sum()

EmpID                       0
Age                         0
AgeGroup                    0
Attrition                   0
BusinessTravel              0
DailyRate                   0
Department                  0
DistanceFromHome            0
Education                   0
EducationField              0
EmployeeCount               0
EmployeeNumber              0
EnvironmentSatisfaction     0
Gender                      0
HourlyRate                  0
JobInvolvement              0
JobLevel                    0
JobRole                     0
JobSatisfaction             0
MaritalStatus               0
MonthlyIncome               0
SalarySlab                  0
MonthlyRate                 0
NumCompaniesWorked          0
Over18                      0
OverTime                    0
PercentSalaryHike           0
PerformanceRating           0
RelationshipSatisfaction    0
StandardHours               0
StockOptionLevel            0
TotalWorkingYears           0
TrainingTimesLastYear       0
WorkLifeBa

In [6]:
# Loại bỏ các cột không có giá trị phân tích
drop_cols = [
    'EmpID',
    'EmployeeNumber',
    'EmployeeCount',
    'Over18',
    'StandardHours'
]

df.drop(columns=drop_cols, inplace=True)
df.shape

(1480, 33)

In [7]:
# Phân tách biến mục tiêu (Target)
target_col = 'Attrition'

y = df[target_col]
X = df.drop(columns=[target_col])

In [8]:
# Mã hóa biến mục tiêu
le_target = LabelEncoder()
y_encoded = le_target.fit_transform(y)

pd.Series(y_encoded).value_counts()

0    1242
1     238
Name: count, dtype: int64

In [9]:
# Xác định biến phân loại & biến số
categorical_cols = X.select_dtypes(include='object').columns.tolist()
numerical_cols = X.select_dtypes(exclude='object').columns.tolist()

categorical_cols, numerical_cols

(['AgeGroup',
  'BusinessTravel',
  'Department',
  'EducationField',
  'Gender',
  'JobRole',
  'MaritalStatus',
  'SalarySlab',
  'OverTime'],
 ['Age',
  'DailyRate',
  'DistanceFromHome',
  'Education',
  'EnvironmentSatisfaction',
  'HourlyRate',
  'JobInvolvement',
  'JobLevel',
  'JobSatisfaction',
  'MonthlyIncome',
  'MonthlyRate',
  'NumCompaniesWorked',
  'PercentSalaryHike',
  'PerformanceRating',
  'RelationshipSatisfaction',
  'StockOptionLevel',
  'TotalWorkingYears',
  'TrainingTimesLastYear',
  'WorkLifeBalance',
  'YearsAtCompany',
  'YearsInCurrentRole',
  'YearsSinceLastPromotion',
  'YearsWithCurrManager'])

In [10]:
# One-Hot Encoding cho biến phân loại
X_encoded = pd.get_dummies(
    X,
    columns=categorical_cols,
    drop_first=True
)

X_encoded.shape

(1480, 52)

In [11]:
# Chuẩn hóa biến số
scaler = StandardScaler()

X_scaled = X_encoded.copy()
X_scaled[numerical_cols] = scaler.fit_transform(
    X_scaled[numerical_cols]
)

X_scaled.describe()

,Age,DailyRate,DistanceFromHome,Education,EnvironmentSatisfaction,HourlyRate,JobInvolvement,JobLevel,JobSatisfaction,MonthlyIncome,MonthlyRate,NumCompaniesWorked,PercentSalaryHike,PerformanceRating,RelationshipSatisfaction,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
count,1.480000e+03,1.480000e+03,1.480000e+03,1.480000e+03,1.480000e+03,1.480000e+03,1.480000e+03,1.480000e+03,1.480000e+03,1.480000e+03,1.480000e+03,1480.000000,1.480000e+03,1.480000e+03,1.480000e+03,1.480000e+03,1480.000000,1.480000e+03,1.480000e+03,1.480000e+03,1.480000e+03,1480.000000,1.480000e+03
mean,1.536309e-16,5.281061e-17,8.281664e-17,1.728347e-16,-1.080217e-16,-3.048612e-16,-1.440289e-16,1.920386e-16,-7.741555e-17,-3.840772e-17,-7.441495e-17,0.000000,1.929388e-16,-1.140229e-16,1.248251e-16,-1.920386e-17,0.000000,-2.520506e-17,1.860374e-16,-1.920386e-17,-4.800964e-17,0.000000,-1.920386e-17
std,1.000338e+00,1.000338e+00,1.000338e+00,1.000338e+00,1.000338e+00,1.000338e+00,1.000338e+00,1.000338e+00,1.000338e+00,1.000338e+00,1.000338e+00,1.000338,1.000338e+00,1.000338e+00,1.000338e+00,1.000338e+00,1.000338,1.000338e+00,1.000338e+00,1.000338e+00,1.000338e+00,1.000338,1.000338e+00
min,-2.073050e+00,-1.735485e+00,-1.011296e+00,-1.867028e+00,-1.578749e+00,-1.763918e+00,-2.426786e+00,-9.635038e-01,-1.562835e+00,-1.169689e+00,-1.716604e+00,-1.077773,-1.152167e+00,-4.256351e-01,-1.579823e+00,-9.313751e-01,-1.452292,-2.171739e+00,-2.491296e+00,-1.146108e+00,-1.169741e+00,-0.678138,-1.167035e+00
25%,-7.580502e-01,-8.347200e-01,-8.882711e-01,-8.899413e-01,-6.631734e-01,-8.781518e-01,-1.023800e+00,-9.635038e-01,-6.568436e-01,-7.624995e-01,-8.787293e-01,-0.676691,-8.785017e-01,-4.256351e-01,-6.552924e-01,-9.313751e-01,-0.679916,-6.193731e-01,-1.076439e+00,-6.555820e-01,-6.164601e-01,-0.678138,-5.942572e-01
50%,-1.005501e-01,-3.435462e-03,-2.731479e-01,8.714559e-02,2.524018e-01,7.614129e-03,3.791852e-01,-5.869058e-02,2.491476e-01,-3.345595e-01,-1.103581e-02,-0.275608,-3.311716e-01,-4.256351e-01,2.692384e-01,2.447641e-01,-0.165000,1.568100e-01,3.384185e-01,-3.285643e-01,-3.398194e-01,-0.367412,-3.078682e-01
75%,6.665333e-01,8.824409e-01,5.880245e-01,1.064233e+00,1.167977e+00,8.441708e-01,3.791852e-01,8.461226e-01,1.155139e+00,3.998499e-01,8.667144e-01,0.526556,7.634884e-01,-4.256351e-01,1.193769e+00,2.447641e-01,0.478647,1.568100e-01,3.384185e-01,3.254709e-01,7.667432e-01,0.254039,8.376878e-01
max,2.529450e+00,1.731096e+00,2.433394e+00,2.041319e+00,1.167977e+00,1.680727e+00,1.782171e+00,2.655749e+00,1.155139e+00,2.871878e+00,1.786379e+00,2.531966,2.679143e+00,2.349431e+00,1.193769e+00,2.597042e+00,3.696877,2.485359e+00,1.753276e+00,5.394244e+00,3.809790e+00,3.982751,3.701578e+00


In [12]:
# Gộp lại dữ liệu hoàn chỉnh
df_processed = X_encoded.copy()
df_processed['Attrition'] = y_encoded

df_processed.head()

,Age,DailyRate,DistanceFromHome,Education,EnvironmentSatisfaction,HourlyRate,JobInvolvement,JobLevel,JobSatisfaction,MonthlyIncome,MonthlyRate,NumCompaniesWorked,PercentSalaryHike,PerformanceRating,RelationshipSatisfaction,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager,AgeGroup_26-35,AgeGroup_36-45,AgeGroup_46-55,AgeGroup_55+,BusinessTravel_TravelRarely,BusinessTravel_Travel_Frequently,BusinessTravel_Travel_Rarely,Department_Research & Development,Department_Sales,EducationField_Life Sciences,EducationField_Marketing,EducationField_Medical,EducationField_Other,EducationField_Technical Degree,Gender_Male,JobRole_Human Resources,JobRole_Laboratory Technician,JobRole_Manager,JobRole_Manufacturing Director,JobRole_Research Director,JobRole_Research Scientist,JobRole_Sales Executive,JobRole_Sales Representative,MaritalStatus_Married,MaritalStatus_Single,SalarySlab_15k+,SalarySlab_5k-10k,SalarySlab_Upto 5k,OverTime_Yes,Attrition
0,18,230,3,3,3,54,3,1,3,1420,25233,1,13,3,3,0,0,2,3,0,0,0,0.0,False,False,False,False,False,False,True,True,False,True,False,False,False,False,True,False,True,False,False,False,False,False,False,False,True,False,False,True,False,1
1,18,812,10,3,4,69,2,1,3,1200,9724,1,12,3,1,0,0,2,3,0,0,0,0.0,False,False,False,False,False,False,True,False,True,False,False,True,False,False,False,False,False,False,False,False,False,False,True,False,True,False,False,True,False,0
2,18,1306,5,3,2,69,3,1,2,1878,8059,1,14,3,4,0,0,3,3,0,0,0,0.0,False,False,False,False,False,True,False,False,True,False,True,False,False,False,True,False,False,False,False,False,False,False,True,False,True,False,False,True,True,1
3,18,287,5,2,2,73,3,1,4,1051,13493,1,15,3,4,0,0,2,3,0,0,0,0.0,False,False,False,False,False,False,False,True,False,True,False,False,False,False,True,False,False,False,False,False,True,False,False,False,True,False,False,True,False,0
4,18,247,8,1,3,80,3,1,3,1904,13556,1,12,3,4,0,0,0,3,0,0,0,0.0,False,False,False,False,False,False,False,True,False,False,False,True,False,False,True,False,True,False,False,False,False,False,False,False,True,False,False,True,False,1


In [13]:
import os

OUTPUT_DIR = "../data/processed"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ===== File cho Apriori =====
df_processed.to_csv(
    os.path.join(OUTPUT_DIR, "hr_processed.csv"),
    index=False
)
print("Saved Apriori file: hr_processed.csv")

# ===== File cho ML / Clustering =====
df_ml = X_scaled.copy()
df_ml["Attrition"] = y_encoded

df_ml.to_csv(
    os.path.join(OUTPUT_DIR, "hr_processed_ml.csv"),
    index=False
)
print("Saved ML file: hr_processed_ml.csv")

Saved Apriori file: hr_processed.csv
Saved ML file: hr_processed_ml.csv


## Summary

Sau bước preprocessing:
- Dữ liệu không còn missing values
- Các cột không mang giá trị phân tích đã được loại bỏ
- Biến phân loại đã được mã hóa
- Biến số đã được chuẩn hóa
- Dữ liệu sẵn sàng cho các bước:
  - Khai phá luật kết hợp
  - Phân cụm
  - Phân lớp & bán giám sát